# nn-parameter-wrap — worked example 3: frozen Parameter stays visible

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `nn-parameter-wrap`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`nn.Parameter(tensor, requires_grad=False)` registers a frozen weight. It still appears in `.parameters()` and `.state_dict()`, but autograd never populates its `.grad`. This is the mechanism behind freezing a backbone in transfer learning.

## Worked solution

We build `MixedScaler` with two same-shaped Parameters: a trainable `gain` and a frozen `shift` created with `requires_grad=False`. The `forward` returns `x * gain + shift`. To demonstrate the freezing, we run a forward on an input requiring grad, sum the output, and call `.backward()`. Afterward, `gain.grad` is a real tensor while `shift.grad` is `None`, because autograd skipped the frozen Parameter. Crucially, both still appear in `.parameters()` and `.state_dict()`, since visibility is independent of `requires_grad`. We print both grads and the state-dict keys.

In [ ]:
import torch as t
import torch.nn as nn

class MixedScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.gain = nn.Parameter(t.ones(3))
        self.shift = nn.Parameter(t.ones(3), requires_grad=False)
    def forward(self, x):
        return x * self.gain + self.shift

mod = MixedScaler()
x = t.randn(3, requires_grad=True)
mod(x).sum().backward()
print('gain.grad is None:', mod.gain.grad is None)
print('shift.grad is None:', mod.shift.grad is None)
print('state_dict keys:', list(mod.state_dict().keys()))